# 06 - Model training

Ten regression families trained with nested cross-validation on the frozen Phase 2 dataset.
Every data-dependent step (imputation, scaling) lives inside a scikit-learn `Pipeline`, so it is fitted on training rows only.

In [1]:
import json, sys, pathlib
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

METRICS = ROOT / "evaluation" / "metrics"
PRED = ROOT / "evaluation" / "predictions"
PLOTS = ROOT / "evaluation" / "plots"
ERRORS = ROOT / "evaluation" / "error_analysis"

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

def show(path):
    try:
        from IPython.display import Image, display
        display(Image(filename=str(path)))
    except Exception:
        print("plot written to", path)


## Model families and search spaces

Compact grids: no family searches more than 12 candidates, which is appropriate for 640 districts.

In [2]:
from src.models.registry import build_models, describe_models

models = build_models()
pd.DataFrame(describe_models(models))[["model", "n_configs", "scaling", "complexity_rank"]].to_string(index=False)

'                 model  n_configs  scaling  complexity_rank\n            dummy_mean          1    False                1\n                 ridge          5     True                2\n           elastic_net         12     True                3\n                   knn          8     True                3\n         decision_tree         12    False                4\n         random_forest          8    False                6\n           extra_trees          8    False                6\n     gradient_boosting          8    False                7\nhist_gradient_boosting          8    False                7\n               svr_rbf          6     True                5'

## The three evaluation matrices

A on 640 districts, A restricted to the 628 districts with a D5 household denominator, and B on those same 628.

In [3]:
from src.training.run_nested_cv import build_configurations

configs = build_configurations(["a640", "a628", "b628"])
for cid, cfg in configs.items():
    print(f"{cfg['label']:<20} districts={len(cfg['y']):>3}  features={cfg['n_features']:>2}  states={len(set(cfg['groups']))}")

A_primary_640        districts=640  features=43  states=35
A_restricted_628     districts=628  features=43  states=35
B_augmented_628      districts=628  features=77  states=35


## Outer folds are grouped by state

The `overlap` column must be zero: a state can never appear in both the training and held-out part of a fold.

In [4]:
from src.training.nested_cv import build_outer_folds

groups = configs["a640"]["groups"]
rows = []
for i, (tr, te) in enumerate(build_outer_folds(groups, "grouped"), start=1):
    rows.append({
        "fold": i,
        "n_train": len(tr),
        "n_test": len(te),
        "states_train": len(set(groups[tr])),
        "states_test": len(set(groups[te])),
        "overlap": len(set(groups[tr]) & set(groups[te])),
    })
pd.DataFrame(rows)

,fold,n_train,n_test,states_train,states_test,overlap
0,1,512,128,29,6,0
1,2,512,128,28,7,0
2,3,512,128,28,7,0
3,4,512,128,27,8,0
4,5,512,128,28,7,0


## Learned transforms are inside the pipeline

The imputer is kept even though the evaluated matrices are complete, so no re-run can bypass in-fold fitting.

In [5]:
for name in ["ridge", "random_forest", "svr_rbf"]:
    print(name)
    for step, obj in models[name].estimator.steps:
        print(f"   {step:<9} {obj}")

ridge
   imputer   SimpleImputer(strategy='median')
   scaler    StandardScaler()
   model     Ridge()
random_forest
   imputer   SimpleImputer(strategy='median')
   model     RandomForestRegressor(n_estimators=200, n_jobs=1, random_state=42)
svr_rbf
   imputer   SimpleImputer(strategy='median')
   scaler    StandardScaler()
   model     SVR()


## Nested-CV results

Grouped by state (primary). `r2_mean` is reported unclipped, so negative values stay visible.

In [6]:
results = pd.read_csv(METRICS / "phase3_nested_cv_results.csv")
results.groupby("configuration")[["mae_mean", "r2_mean", "spearman_mean"]].agg(["min", "median"]).round(3)

mae_mean        r2_mean        spearman_mean       
                      min median     min median           min median
configuration                                                       
A_primary_640       2.029  2.176  -0.336  0.042         0.275  0.448
A_restricted_628    2.089  2.161  -0.116  0.078         0.264  0.450
B_augmented_628     1.725  1.852  -0.184  0.336         0.375  0.626

## Hyperparameters chosen in the inner loop

The inner 4-fold search is re-run inside every outer fold, so the chosen configuration can differ by fold. That variation is itself information about tuning stability.

In [7]:
payload = json.loads((METRICS / "phase3_best_parameters.json").read_text(encoding="utf-8"))
selected = pd.read_csv(METRICS / "phase3_selected_models.csv")
for _, row in selected.iterrows():
    per_fold = payload["grouped_best_parameters"]["best_params"][row["configuration"]][row["model"]]["per_fold"]
    print(f"{row['configuration']} / {row['model']}")
    for index, params in enumerate(per_fold, start=1):
        print(f"   fold {index}: {params}")

A_primary_640 / svr_rbf
   fold 1: {'model__C': 1.0, 'model__epsilon': 0.5, 'model__gamma': 'scale'}
   fold 2: {'model__C': 10.0, 'model__epsilon': 0.5, 'model__gamma': 'scale'}
   fold 3: {'model__C': 1.0, 'model__epsilon': 0.5, 'model__gamma': 'scale'}
   fold 4: {'model__C': 10.0, 'model__epsilon': 0.5, 'model__gamma': 'scale'}
   fold 5: {'model__C': 1.0, 'model__epsilon': 0.1, 'model__gamma': 'scale'}
A_restricted_628 / extra_trees
   fold 1: {'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1}
   fold 2: {'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 3}
   fold 3: {'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1}
   fold 4: {'model__max_depth': 10, 'model__max_features': 0.5, 'model__min_samples_leaf': 3}
   fold 5: {'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1}
B_augmented_628 / svr_rbf
   fold 1: {'model__C': 10.0, 'model__epsilon'

## Leakage guard

No approved predictor may come from a forbidden source, and the feature matrices must equal the approved manifest exactly.

In [8]:
from src.features.predictors import FORBIDDEN_SOURCES

manifest = pd.read_csv(ROOT / "datasets" / "processed" / "feature_manifest.csv")
used = {c for v in manifest[manifest["retained"]]["source_columns"] for c in str(v).split("|")}
print("forbidden sources used:", sorted(used & FORBIDDEN_SOURCES))
print("features per configuration:", results.groupby("configuration")["n_features"].first().to_dict())
print("models per configuration:", results.groupby("configuration")["model"].nunique().to_dict())

forbidden sources used: []
features per configuration: {'A_primary_640': 43, 'A_restricted_628': 43, 'B_augmented_628': 77}
models per configuration: {'A_primary_640': 10, 'A_restricted_628': 10, 'B_augmented_628': 10}
